# RAG from Scratch: PDF Question Answering

A compact Retrieval-Augmented Generation (RAG) pipeline built as part of an AI bootcamp exercise. The notebook loads a PDF, prepares it for semantic search, retrieves relevant passages, and generates a grounded answer with an LLM.

### Pipeline

```text
PDF → Text → Overlapping Chunks → Embeddings → Similarity Search → Retrieved Context → LLM Answer
```

The implementation intentionally uses NumPy and cosine similarity instead of a vector database so the core retrieval mechanics remain visible.


## Learning outcomes

This notebook demonstrates how to:

- extract text from a PDF.
- split long text into overlapping chunks.
- create embeddings for document chunks and user queries.
- rank chunks with cosine similarity.
- augment an LLM prompt with retrieved context.
- generate answers that stay grounded in the source.
- perform a simple grounding check.


## Implementation notes

- The OpenAI API key is entered at runtime with `getpass` and is not stored in the notebook.
- Retrieval uses an in-memory NumPy embedding matrix for clarity.
- `TOP_K = 3` is used to keep the retrieved context focused.
- The final grounding metric is educational and lexical; it is not a substitute for full RAG evaluation.


## 1. Setup


In [5]:
!pip install -q -U openai pypdf numpy

from getpass import getpass
from pathlib import Path
import re
import numpy as np
from openai import OpenAI
from pypdf import PdfReader

api_key = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=api_key)
CHAT_MODEL = "gpt-5-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

print("Setup complete")

Enter your OpenAI API key: ··········
Setup complete


## 2. Load the source document

The PDF is converted into plain text so it can be chunked, embedded, and searched.


In [ ]:
# TODO 1: Read every page from PDF_PATH and combine the extracted text.
# Save the result in document_text and print its character count.
PDF_PATH = "rag_llm_practice.pdf"
reader = PdfReader(PDF_PATH)
document_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        document_text += text + "\n"
print("Document characters:", len(document_text) if document_text else 0)

Document characters: 4588


## 3. Chunk the document

The extracted text is split into overlapping chunks. Overlap helps preserve context when an idea crosses a chunk boundary.


In [10]:
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# TODO 2: Write chunk_text(text, chunk_size, overlap).
# Return a list of non-empty overlapping text chunks.
def chunk_text(text, chunk_size=900, overlap=150):
    chunks = []

    step = chunk_size - overlap

    for start in range(0, len(text), step):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

    return chunks

chunks = chunk_text(document_text, CHUNK_SIZE, CHUNK_OVERLAP)

print("Number of chunks:", len(chunks))
print("First chunk preview:\n", chunks[0][:500])

Number of chunks: 7
First chunk preview:
 RAG and Large Language Models - Practice
 Document
This short reference document is designed for practicing a Retrieval-Augmented Generation (RAG)
pipeline. It contains clear facts that can be extracted, chunked, embedded, retrieved, and used to
answer questions.
1. What is Retrieval-Augmented Generation?
Retrieval-Augmented Generation, commonly called RAG, is a technique that combines information
retrieval with a large language model. Instead of relying only on the model's internal knowledge, a


## 4. Create embeddings and an in-memory vector index

Each chunk is converted into an embedding vector. The vectors are stored in NumPy, and cosine similarity is used to compare them with the embedded user query.


In [11]:
# TODO 3: Complete embed_texts so it returns one list of embedding vectors per input text.
def embed_texts(texts):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

# TODO 4: Complete cosine_similarity_matrix.
# It should return the cosine similarity between one query vector and every row in matrix.
def cosine_similarity_matrix(query_vector, matrix):
    query_vector = np.array(query_vector)

    dot_products = matrix @ query_vector

    matrix_norms = np.linalg.norm(matrix, axis=1)
    query_norm = np.linalg.norm(query_vector)

    similarities = dot_products / (matrix_norms * query_norm)

    return similarities

chunk_embeddings = np.array(embed_texts(chunks), dtype=np.float32)
print("Embedding matrix shape:", chunk_embeddings.shape)

Embedding matrix shape: (7, 1536)


## 5. Retrieve relevant context

The user question is embedded with the same embedding model, compared against all chunk embeddings, and the top-scoring chunks are selected as retrieval context.


In [12]:
question = "What is the difference between RAG and fine-tuning, and when should each be used?"
TOP_K = 3

query_embedding = embed_texts([question])[0]

similarities = cosine_similarity_matrix(query_embedding, chunk_embeddings)

top_indices = np.argsort(similarities)[::-1][:TOP_K]

retrieved_chunks = [chunks[i] for i in top_indices]
retrieved_scores = [similarities[i] for i in top_indices]

for rank, (score, chunk) in enumerate(zip(retrieved_scores, retrieved_chunks), start=1):
    print(f"[{rank}] score={score:.3f}\n{chunk[:400]}\n")

[1] score=0.450
ontext does not contain enough information.
The language model then performs inference and generates an answer. In a RAG system, the model
itself does not normally search the vector database. The application performs retrieval first and then
supplies the retrieved text to the model.
6. Why RAG is Useful
RAG can help reduce hallucination by grounding model responses in external evidence. It also al

[2] score=0.436
RAG and Large Language Models - Practice
 Document
This short reference document is designed for practicing a Retrieval-Augmented Generation (RAG)
pipeline. It contains clear facts that can be extracted, chunked, embedded, retrieved, and used to
answer questions.
1. What is Retrieval-Augmented Generation?
Retrieval-Augmented Generation, commonly called RAG, is a technique that combines information

[3] score=0.428
Problems can occur during text extraction, chunking,
embedding, retrieval, prompt construction, or generation. For this reason, RAG systems should b

## 6. Generate a grounded answer

The retrieved chunks are joined into context and inserted into an augmented prompt. The model is instructed to answer from the provided context and acknowledge when the source does not contain enough information.


In [14]:
# TODO 6: Build context from retrieved_chunks.
context = "\n\n".join(retrieved_chunks)

# TODO 7: Write an augmented prompt containing the question and context.
augmented_prompt = f"""
Answer the question using only the context below.

If the answer is not supported by the context, say:
"I don't have enough information in the provided context."

Question:
{question}

Context:
{context}
"""

# TODO 8: Call the Responses API and save the answer in answer.
response = client.responses.create(
    model=CHAT_MODEL,
    input=augmented_prompt
)

answer = response.output_text

print("Answer:")
print(answer)

Answer:
What the context supports about RAG
- Retrieval-Augmented Generation (RAG) combines information retrieval with a large language model: the system first searches an external knowledge source, then supplies the retrieved text as context in the model prompt before generation.
- In RAG the application (not the LLM) normally performs the vector-database search and supplies retrieved passages to the model.
- RAG can reduce hallucination by grounding responses in external evidence and lets you use private or frequently updated information without retraining the entire language model.
- Common RAG use cases: question answering over company documents, legal documents, technical manuals, policies, research papers, and internal knowledge bases.
- Limitations: RAG does not guarantee correct answers; errors can arise in extraction, chunking, embedding, retrieval, prompt construction, or generation; evaluate the whole pipeline (retrieval relevance, answer correctness, faithfulness, latency, 

## 7. Evaluate grounding

A lightweight lexical check measures how many distinctive terms in the generated answer also appear in the retrieved context. This is useful for inspection, but stronger RAG systems should also evaluate retrieval relevance, faithfulness, correctness, latency, and cost.


In [15]:
# TODO 9: Normalize text and calculate how many distinctive context terms appear in the answer.
def normalize_words(text):
    return set(re.findall(r"[a-zA-Z]{4,}", text.lower()))

context_terms = normalize_words(context or "")
answer_terms = normalize_words(answer or "")
overlap = context_terms & answer_terms
grounding_ratio = len(overlap) / max(len(answer_terms), 1)

print(f"Distinctive-term overlap: {len(overlap)}")
print(f"Simple grounding ratio: {grounding_ratio:.2%}")
print("Retrieved context used:", bool(context))

Distinctive-term overlap: 71
Simple grounding ratio: 80.68%
Retrieved context used: True


## 8. Conceptual reflection

Key questions for understanding the pipeline:

1. What roles do indexing, chunking, retrieval, and augmentation play in RAG?
2. Why can poor retrieval produce a poor answer even when the language model is capable?
3. How does semantic vector search differ from traditional keyword matching?
4. Why should a grounded system acknowledge missing evidence instead of answering from unsupported knowledge?


## What this notebook demonstrates

- End-to-end RAG flow from document ingestion to grounded generation
- Overlapping text chunking
- Embedding-based semantic retrieval
- Cosine-similarity ranking and top-k selection
- Context augmentation for LLM generation
- Basic grounding inspection
